In [1]:
from datasets import load_dataset
from itertools import chain
import pandas as pd

# splits = ["2020", "2021", "2022", "2023", "2024", "2025", "2026"]
splits = ["2020", "2021", "2022", "2023"]

columns_to_keep = [
    'submission_id',
    'year',
    'openreview_link',
    'pdf_download_link',
    'title',
    'original_abstract',
    'original_reviews',
    'normalized_reviews',
    'original_metareview',
    'technical_indicators'
]

streams = [
    load_dataset(
        "skonan/iclr-reviews-2020-2026",
        split=s,
        streaming=True
    )
    for s in splits
]

df_with_rebuttal = pd.DataFrame(
    {col: row.get(col) for col in columns_to_keep}
    for row in chain.from_iterable(streams)
)

df_with_rebuttal.head() # dataframe from auto reviewer

,submission_id,year,openreview_link,pdf_download_link,title,original_abstract,original_reviews,normalized_reviews,original_metareview,technical_indicators
0,ryxz8CVYDH,2020,https://openreview.net/forum?id=ryxz8CVYDH,https://openreview.net/pdf/07a4b4b413b37f2c43c...,Learning to Learn by Zeroth-Order Oracle,"In the learning to learn (L2L) framework, we c...","[{""rating"": ""6: Weak Accept"", ""review"": ""The p...","[""{\""summary\"":\""The paper introduces a zeroth...","{""decision"": ""Accept (Poster)"", ""comment"": ""Th...","{""binary_decision"": ""accept"", ""specific_decisi..."
1,ryxyCeHtPB,2020,https://openreview.net/forum?id=ryxyCeHtPB,https://openreview.net/pdf/84d13359f1520c521f1...,"Pay Attention to Features, Transfer Learn Fast...",Deep convolutional neural networks are now wid...,"[{""rating"": ""6: Weak Accept"", ""review"": "" This...","[""{\""summary\"":\""The paper introduces Attentiv...","{""decision"": ""Accept (Poster)"", ""comment"": ""Th...","{""binary_decision"": ""accept"", ""specific_decisi..."
2,ryxtWgSKPB,2020,https://openreview.net/forum?id=ryxtWgSKPB,https://openreview.net/pdf/e90a912a0f6b4596f6b...,Quantum Optical Experiments Modeled by Long Sh...,We demonstrate how machine learning is able to...,"[{""rating"": ""3: Weak Reject"", ""review"": ""This ...","[""{\""summary\"":\""The paper proposes using mach...","{""decision"": ""Reject"", ""comment"": ""The paper p...","{""binary_decision"": ""reject"", ""specific_decisi..."
3,ryxtCpNtDS,2020,https://openreview.net/forum?id=ryxtCpNtDS,https://openreview.net/pdf/7d53de57a3ebd9cc3bb...,Autoencoders and Generative Adversarial Networ...,We introduce a novel synthetic oversampling me...,"[{""rating"": ""3: Weak Reject"", ""review"": ""The p...","[""{\""summary\"":\""The paper addresses an import...","{""decision"": ""Reject"", ""comment"": ""This paper ...","{""binary_decision"": ""reject"", ""specific_decisi..."
4,ryxsUySFwr,2020,https://openreview.net/forum?id=ryxsUySFwr,https://openreview.net/pdf/89002f34556ae166746...,Neural Network Out-of-Distribution Detection f...,Neural network out-of-distribution (OOD) detec...,"[{""rating"": ""3: Weak Reject"", ""review"": ""The a...","[""{\""summary\"":\""The paper presents an empiric...","{""decision"": ""Reject"", ""comment"": ""The paper i...","{""binary_decision"": ""reject"", ""specific_decisi..."


In [2]:
df_with_rebuttal["year"].value_counts()

year
2023    3783
2022    2603
2021    2505
2020    2198
Name: count, dtype: int64

In [3]:
# parquet in iclrdataset
import pandas as pd
import numpy as np

# Load the parquet file
df = pd.read_parquet('../data/iclr26v1.parquet')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total number of papers: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")
print(f"\nShape: {df.shape}")

DATASET OVERVIEW
Total number of papers: 55,906
Number of columns: 10

Shape: (55906, 10)


In [4]:
df.columns

Index(['year', 'id', 'title', 'abstract', 'authors', 'author_ids', 'decision',
       'scores', 'keywords', 'labels'],
      dtype='str')

In [5]:
df_with_rebuttal.columns

Index(['submission_id', 'year', 'openreview_link', 'pdf_download_link',
       'title', 'original_abstract', 'original_reviews', 'normalized_reviews',
       'original_metareview', 'technical_indicators'],
      dtype='str')

In [6]:
# Inner join: df has `id` column, df_with_rebuttal has `submission_id` column
# Common columns: 'year' and 'title' - we'll keep them from df and drop from df_with_rebuttal

# Identify common columns (excluding the join keys)
common_cols = set(df.columns) & set(df_with_rebuttal.columns)
# Remove the join key columns from common columns
common_cols = common_cols - {'id', 'submission_id'}
print(f"Common columns (will keep from df): {common_cols}")

# Drop common columns from df_with_rebuttal to avoid _x/_y suffixes
df_with_rebuttal_clean = df_with_rebuttal.drop(columns=list(common_cols))
print(f"\ndf_with_rebuttal columns after dropping duplicates: {list(df_with_rebuttal_clean.columns)}")

# Merge: use left_on and right_on to specify different column names
df_merged = pd.merge(
    df, 
    df_with_rebuttal_clean, 
    left_on='id',           # Column name in df
    right_on='submission_id',  # Column name in df_with_rebuttal
    how='inner'
)

print(f"\nOriginal df shape: {df.shape}")
print(f"df_with_rebuttal shape: {df_with_rebuttal.shape}")
print(f"Merged df shape: {df_merged.shape}")
print(f"\nColumns in merged df ({len(df_merged.columns)}): {list(df_merged.columns)}")

df_merged.head()

# save to parquet
# df_merged.to_parquet('../data/iclr26v1_with_rebuttal.parquet')
# print(f"\nSaved merged dataframe to '../data/iclr26v1_with_rebuttal.parquet'")


Common columns (will keep from df): {'title', 'year'}

df_with_rebuttal columns after dropping duplicates: ['submission_id', 'openreview_link', 'pdf_download_link', 'original_abstract', 'original_reviews', 'normalized_reviews', 'original_metareview', 'technical_indicators']

Original df shape: (55906, 10)
df_with_rebuttal shape: (11089, 10)
Merged df shape: (11089, 18)

Columns in merged df (18): ['year', 'id', 'title', 'abstract', 'authors', 'author_ids', 'decision', 'scores', 'keywords', 'labels', 'submission_id', 'openreview_link', 'pdf_download_link', 'original_abstract', 'original_reviews', 'normalized_reviews', 'original_metareview', 'technical_indicators']


,year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels,submission_id,openreview_link,pdf_download_link,original_abstract,original_reviews,normalized_reviews,original_metareview,technical_indicators
0,2020,B1e-kxSKDH,Structured Object-Aware Physics Prediction for...,"When humans observe a physical system, they ca...","Jannik Kossen, Karl Stelzner, Marcel Hussing, ...",,Accept (Poster),"[6, 6, 6]","[self-supervised learning, probabilistic deep ...",autoencoders,B1e-kxSKDH,https://openreview.net/forum?id=B1e-kxSKDH,https://openreview.net/pdf/28defda144549878c89...,"When humans observe a physical system, they ca...","[{""rating"": ""6: Weak Accept"", ""review"": ""This ...","[""{\""summary\"":\""The paper presents a structur...","{""decision"": ""Accept (Poster)"", ""comment"": ""Th...","{""binary_decision"": ""accept"", ""specific_decisi..."
1,2020,B1e3OlStPB,DeepSphere: a graph-based spherical CNN,Designing a convolution for a spherical neural...,"Michaël Defferrard, Martino Milani, Frédérick ...",,Accept (Spotlight),"[8, 6, 6]","[spherical cnns, graph neural networks, geomet...",graphs,B1e3OlStPB,https://openreview.net/forum?id=B1e3OlStPB,https://openreview.net/pdf/1557f75b9013011ccab...,Designing a convolution for a spherical neural...,"[{""rating"": ""6: Weak Accept"", ""review"": ""In th...","[""{\""summary\"":\""The paper studies CNNs specia...","{""decision"": ""Accept (Spotlight)"", ""comment"": ...","{""binary_decision"": ""accept"", ""specific_decisi..."
2,2020,B1e5TA4FPr,Pareto Optimality in No-Harm Fairness,Common fairness definitions in machine learnin...,"Natalia Martinez, Martin Bertran, Guillermo Sa...",,Reject,"[3, 3, 3]","[fairness, fairness in machine learning, no-ha...",fairness,B1e5TA4FPr,https://openreview.net/forum?id=B1e5TA4FPr,https://openreview.net/pdf/141431619907795b4f2...,Common fairness definitions in machine learnin...,"[{""rating"": ""3: Weak Reject"", ""review"": ""This ...","[""{\""summary\"":\""The paper proposes a fairness...","{""decision"": ""Reject"", ""comment"": ""This manusc...","{""binary_decision"": ""reject"", ""specific_decisi..."
3,2020,B1e9Y2NYvS,On Robustness of Neural Ordinary Differential ...,Neural ordinary differential equations (ODEs) ...,"Hanshu YAN, Jiawei DU, Vincent TAN, Jiashi FENG",,Accept (Spotlight),"[6, 8, 6]",[neural ode],unlabeled,B1e9Y2NYvS,https://openreview.net/forum?id=B1e9Y2NYvS,https://openreview.net/pdf/48923b13b7666cc0e57...,Neural ordinary differential equations (ODEs)...,"[{""rating"": ""6: Weak Accept"", ""review"": ""This ...","[""{\""summary\"":\""The paper investigates the ro...","{""decision"": ""Accept (Spotlight)"", ""comment"": ...","{""binary_decision"": ""accept"", ""specific_decisi..."
4,2020,B1eB5xSFvr,DiffTaichi: Differentiable Programming for Phy...,"We present DiffTaichi, a new differentiable pr...","Yuanming Hu, Luke Anderson, Tzu-Mao Li, Qi Sun...",,Accept (Poster),"[6, 3, 6]","[differentiable programming, robotics, optimal...",unlabeled,B1eB5xSFvr,https://openreview.net/forum?id=B1eB5xSFvr,https://openreview.net/pdf/bee5278794e8dc780a2...,"We present DiffTaichi, a new differentiable pr...","[{""rating"": ""6: Weak Accept"", ""review"": ""This ...","[""{\""summary\"":\""The reviewer acknowledges the...","{""decision"": ""Accept (Poster)"", ""comment"": ""Th...","{""binary_decision"": ""accept"", ""specific_decisi..."


In [7]:
df_merged['labels'].value_counts() # 74 papers on optimization in 2020, 

labels
unlabeled                     5032
RL                             618
adversarial                    418
graphs                         386
optimization                   348
language models                273
transformers                   251
self-supervised learning       226
GANs                           187
transfer learning              177
autoencoders                   172
meta learning                  164
federated learning             141
out-of-distribution            141
continual learning             133
CNNs                           123
RNNs                           112
multi-agent RL                 111
LLMs                           107
offline RL                     101
robustness                      99
privacy                         93
compression                     90
explainability                  86
few-shot learning               81
causality                       80
semi-supervised learning        75
neural architecture search      74
object detect

In [8]:
df['decision'].value_counts()

decision
                            19673
Reject                      17096
Withdrawn                    7594
Accept (Poster)              6173
Accept (poster)              1809
Accept: poster               1202
Accept (Spotlight)            775
Accept (Oral)                 383
Accept (spotlight)            366
Accept: notable-top-25%       280
Desk rejected                 194
Invite to Workshop Track      136
Accept: notable-top-5%         91
Accept (oral)                  86
Accept (Talk)                  48
Name: count, dtype: int64

In [9]:
df_merged['decision'].value_counts()

decision
Reject                     6912
Accept (Poster)            2066
Accept: poster             1201
Accept (Spotlight)          388
Accept: notable-top-25%     280
Accept (Oral)               105
Accept: notable-top-5%       90
Accept (Talk)                47
Name: count, dtype: int64

In [13]:
df['labels'].unique() # all the unique areas in ICLR

<ArrowStringArray>
[         'transfer learning',            'language models',
                  'unlabeled',               'optimization',
   'semi-supervised learning',                         'RL',
        'multi-task learning',                       'GANs',
                       'RNNs',                 'clustering',
            'code generation',               'autoencoders',
          'few-shot learning',             'multi-agent RL',
                       'LLMs',               'neuroscience',
                'compression',         'continual learning',
                       'CNNs',                     'speech',
                'adversarial',                    'privacy',
         'imitation learning',               'transformers',
                     'graphs',              'meta learning',
 'neural architecture search',                 'robustness',
          'optimal transport',           'object detection',
        'out-of-distribution',           'interpretability',
     

In [14]:
import pandas as pd
import numpy as np

years = [2020, 2021, 2022, 2023]
topics = ["optimization", "language models"]
N_PER_GROUP = 10
RANDOM_STATE = 42

def normalize_decision(decision: str) -> str:
    if decision is None or (isinstance(decision, float) and np.isnan(decision)):
        return "unknown"
    d = str(decision).strip().lower()
    if "withdraw" in d:
        return "withdrawn"
    if "reject" in d:
        return "rejected"
    if "accept" in d or "oral" in d or "spotlight" in d or "poster" in d:
        return "accepted"
    return "unknown"

def add_decision_group(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "decision" in out.columns:
        out["decision_group"] = out["decision"].apply(normalize_decision)
    elif "final_decision" in out.columns:
        out["decision_group"] = out["final_decision"].apply(normalize_decision)
    else:
        out["decision_group"] = "unknown"
    return out

def stratified_sample(group_df: pd.DataFrame, n: int, seed: int, include_withdrawn: bool) -> pd.DataFrame:
    """
    include_withdrawn=False -> mix accepted/rejected only
    include_withdrawn=True  -> mix accepted/rejected/withdrawn
    """
    if len(group_df) == 0:
        return group_df

    df = add_decision_group(group_df)

    accepted = df[df["decision_group"] == "accepted"]
    rejected = df[df["decision_group"] == "rejected"]
    withdrawn = df[df["decision_group"] == "withdrawn"] if include_withdrawn else df.iloc[0:0]

    if include_withdrawn:
        quotas = {"accepted": 4, "rejected": 4, "withdrawn": 2}
        order = ["accepted", "rejected", "withdrawn"]
        pools = {"accepted": accepted, "rejected": rejected, "withdrawn": withdrawn}
        fallback_concat = [accepted, rejected, withdrawn, df]
    else:
        # 5/5 split; if one side is short, remainder gets filled from the other
        quotas = {"accepted": 5, "rejected": 5}
        order = ["accepted", "rejected"]
        pools = {"accepted": accepted, "rejected": rejected}
        fallback_concat = [accepted, rejected, df]

    selected_parts = []
    remaining = n

    # Pass 1: hit quotas as much as possible
    for k in order:
        take = min(quotas[k], len(pools[k]), remaining)
        if take > 0:
            selected_parts.append(pools[k].sample(n=take, random_state=seed))
            remaining -= take

    # Pass 2: fill remaining from anything not already selected (preference order above)
    if remaining > 0:
        selected = pd.concat(selected_parts, ignore_index=False) if selected_parts else df.iloc[0:0]
        selected_idx = set(selected.index)

        fallback = pd.concat(fallback_concat, ignore_index=False).loc[lambda x: ~x.index.isin(selected_idx)]
        if len(fallback) > 0:
            take = min(remaining, len(fallback))
            selected_parts.append(fallback.sample(n=take, random_state=seed + 1))
            remaining -= take

    out = pd.concat(selected_parts, ignore_index=False) if selected_parts else df.iloc[0:0]
    return out.head(n).reset_index(drop=True)

def sample_all(merged: pd.DataFrame, include_withdrawn: bool) -> pd.DataFrame:
    sampled_frames = []
    for year in years:
        for topic in topics:
            subset = merged[(merged["year"] == year) & (merged["labels"] == topic)].copy()
            if len(subset) == 0:
                continue
            n_samples = min(N_PER_GROUP, len(subset))
            sampled = stratified_sample(subset, n=n_samples, seed=RANDOM_STATE, include_withdrawn=include_withdrawn)
            sampled["sample_year"] = year
            sampled["sample_topic"] = topic
            sampled_frames.append(sampled)

    return pd.concat(sampled_frames, ignore_index=True) if sampled_frames else pd.DataFrame()

# --- Two versions ---
sampled_df_with_withdrawn = sample_all(df, include_withdrawn=True)
sampled_df_accept_reject_only = sample_all(df_merged, include_withdrawn=False)

# sampled_df_with_withdrawn.head(20), sampled_df_with_withdrawn.shape
# sampled_df_accept_reject_only.head(20), sampled_df_accept_reject_only.shape

print("Decision distribution for Accept/Reject Only\n", sampled_df_accept_reject_only['decision'].value_counts())
print("\nDecision distribution for Accept/Reject/Waitlist\n",sampled_df_with_withdrawn['decision'].value_counts())

Decision distribution for Accept/Reject Only
 decision
Reject                     40
Accept (Poster)            20
Accept: poster              9
Accept (Spotlight)          7
Accept (Oral)               2
Accept (Talk)               1
Accept: notable-top-25%     1
Name: count, dtype: int64

Decision distribution for Accept/Reject/Waitlist
 decision
Reject                     32
Accept (Poster)            17
Withdrawn                  16
Accept: poster              7
Accept (Spotlight)          5
Accept (Talk)               1
Accept (Oral)               1
Accept: notable-top-25%     1
Name: count, dtype: int64


In [15]:
# Get the paper IDs from the previous cell
accept_reject_paper_ids = sampled_df_accept_reject_only['id'].tolist()
with_withdrawn_paper_ids = sampled_df_with_withdrawn['id'].tolist()
print(f"Fetching reviews for {len(accept_reject_paper_ids)} accept/reject papers and {len(with_withdrawn_paper_ids)} accept/reject/withdrawn papers")
overlap = set(accept_reject_paper_ids) & set(with_withdrawn_paper_ids)
print(f"Overlap of {len(overlap)} papers")

Fetching reviews for 80 accept/reject papers and 80 accept/reject/withdrawn papers
Overlap of 42 papers


In [16]:
import openreview
import pandas as pd
import time

def get_value_api2(field):
    """Extract value from API v2 nested dict format (used by AutoReviewer)"""
    if isinstance(field, dict):
        return field.get('value', field)
    return field

def content_dict_to_text(content_dict):
    """Extract text from content dictionary"""
    main_keys = ["title", "summary", "strengths", "weaknesses", "questions", "comment", "metareview", "rating", "review"]
    text = ""
    for k in main_keys:
        data = content_dict.get(k)
        if data is None:
            continue
        # Handle both old format (direct value) and new format (dict with 'value')
        if isinstance(data, dict) and 'value' in data:
            text += f"\n{k}:\n{data['value']}\n"
        else:
            text += f"\n{k}:\n{str(data)}\n"
    return text.strip()

def get_reviews_for_paper(forum_id, year=2020):
    """
    Fetch reviews for a specific paper using OpenReview client (AutoReviewer approach).
    
    For 2020 (API v1): Uses openreview.Client
    For 2024+ (API v2): Uses openreview.api.OpenReviewClient
    """
    try:
        # Initialize appropriate client based on year (following AutoReviewer approach)
        if year < 2024:
            client = openreview.Client(baseurl='https://api.openreview.net')
        else:
            client = openreview.api.OpenReviewClient(baseurl='https://api2.openreview.net')
        
        # Get all notes in the forum (submission + all replies)
        # This is the proper way according to AutoReviewer
        notes = client.get_notes(forum=forum_id)
        
        reviews = []
        invitation_types = {}
        
        for note in notes:
            # Get invitation - API v1 uses 'invitation' (string attribute), API v2 uses 'invitations' (list attribute)
            if year < 2024:
                invitation = getattr(note, 'invitation', '')
                invitations = [invitation] if invitation else []
            else:
                invitations = getattr(note, 'invitations', [])
                invitation = invitations[0] if invitations else ''
            
            if invitation:
                inv_type = invitation.split("/")[-1]
                invitation_types[inv_type] = invitation_types.get(inv_type, 0) + 1
            
            # Check if it's an Official_Review (following AutoReviewer pattern)
            is_review = False
            if year < 2024:
                is_review = 'Official_Review' in invitation
            else:
                is_review = any('Official_Review' in inv for inv in invitations) if invitations else False
            
            if is_review:
                content = note.content if hasattr(note, 'content') else {}
                
                review_data = {
                    'paper_id': forum_id,
                    'review_id': note.id if hasattr(note, 'id') else '',
                    'review_type': inv_type if invitation else '',
                    'rating': None,
                    'summary': None,
                    'strengths': None,
                    'weaknesses': None,
                    'questions': None,
                    'comment': None,
                    'review': None,  # Full review text
                    'content': None,
                    'cdate': note.cdate if hasattr(note, 'cdate') else None,
                    'writer': note.signatures[0] if hasattr(note, 'signatures') and note.signatures else None,
                }
                
                # Extract fields - handle API v1 (direct) vs API v2 (nested) format
                if year < 2024:
                    # API v1: direct values
                    review_data['rating'] = content.get('rating', '')
                    review_data['review'] = content.get('review', '')
                    for field in ['summary', 'strengths', 'weaknesses', 'questions', 'comment', 'title']:
                        review_data[field] = content.get(field, '')
                else:
                    # API v2: nested {'value': ...} format
                    review_data['rating'] = get_value_api2(content.get('rating', ''))
                    review_data['review'] = get_value_api2(content.get('review', ''))
                    for field in ['summary', 'strengths', 'weaknesses', 'questions', 'comment', 'title']:
                        review_data[field] = get_value_api2(content.get(field, ''))
                
                # Get full content text for reference
                review_data['content'] = content_dict_to_text(content)
                
                reviews.append(review_data)
        
        # Debug output
        if len(reviews) == 0 and len(notes) > 0:
            print(f"  Found {len(notes)} notes but 0 reviews. Invitation types: {invitation_types}", end=' ')
        else:
            print(f"  Found {len(reviews)} reviews", end=' ')
        
        return reviews
    except Exception as e:
        print(f"  Error: {str(e)[:100]}", end=' ')
        return []

# Fetch reviews for all papers
all_reviews = []
print("Fetching reviews using AutoReviewer approach...")
print("=" * 60)

for i, paper_id in enumerate(accept_reject_paper_ids, 1):
    print(f"[{i}/{len(accept_reject_paper_ids)}] Fetching reviews for {paper_id}...", end=' ')
    try:
        reviews = get_reviews_for_paper(paper_id, year=2020)
        all_reviews.extend(reviews)
        print(f"✓")
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
    time.sleep(0.5)  # Small delay to avoid rate limiting (like AutoReviewer)

print(f"\n{'=' * 60}")
print(f"Total accept/reject reviews fetched: {len(all_reviews)}")

# Create DataFrame
if all_reviews:
    reviews_df_accept_reject = pd.DataFrame(all_reviews)
    print(f"\nReviews DataFrame shape: {reviews_df_accept_reject.shape}")
    print(f"\nColumns: {list(reviews_df_accept_reject.columns)}")
    print(f"\nReviews per paper:")
    print(reviews_df_accept_reject['paper_id'].value_counts().sort_index())
else:
    print("\nNo reviews found!")
    reviews_df_accept_reject = pd.DataFrame()

# Fetch reviews for all papers
all_reviews = []
print("Fetching reviews using AutoReviewer approach...")
print("=" * 60)

for i, paper_id in enumerate(with_withdrawn_paper_ids, 1):
    print(f"[{i}/{len(with_withdrawn_paper_ids)}] Fetching reviews for {paper_id}...", end=' ')
    try:
        reviews = get_reviews_for_paper(paper_id, year=2020)
        all_reviews.extend(reviews)
        print(f"✓")
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
    time.sleep(0.5)  # Small delay to avoid rate limiting (like AutoReviewer)

print(f"\n{'=' * 60}")
print(f"Total reviews fetched: {len(all_reviews)}")

# Create DataFrame
if all_reviews:
    reviews_df_with_withdrawn = pd.DataFrame(all_reviews)
    print(f"\nReviews DataFrame shape: {reviews_df_with_withdrawn.shape}")
    print(f"\nColumns: {list(reviews_df_with_withdrawn.columns)}")
    print(f"\nReviews per paper:")
    print(reviews_df_with_withdrawn['paper_id'].value_counts().sort_index())
else:
    print("\nNo reviews found!")
    reviews_df_with_withdrawn = pd.DataFrame()

Fetching reviews using AutoReviewer approach...
[1/80] Fetching reviews for HkgTTh4FDH...   Found 3 reviews ✓
[2/80] Fetching reviews for SJeLIgBKPS...   Found 3 reviews ✓
[3/80] Fetching reviews for HyevIJStwH...   Found 3 reviews ✓
[4/80] Fetching reviews for rkeNfp4tPr...   Found 3 reviews ✓
[5/80] Fetching reviews for B1g5sA4twr...   Found 3 reviews ✓
[6/80] Fetching reviews for SJgn3lBtwH...   Found 3 reviews ✓
[7/80] Fetching reviews for rJeA_aVtPB...   Found 3 reviews ✓
[8/80] Fetching reviews for S1xJFREKvB...   Found 4 reviews ✓
[9/80] Fetching reviews for ryGWhJBtDB...   Found 3 reviews ✓
[10/80] Fetching reviews for HyxehhNtvS...   Found 4 reviews ✓
[11/80] Fetching reviews for HkxARkrFwB...   Found 3 reviews ✓
[12/80] Fetching reviews for SylKikSYDH...   Found 3 reviews ✓
[13/80] Fetching reviews for BkxRRkSKwr...   Found 3 reviews ✓
[14/80] Fetching reviews for ryxgsCVYPr...   Found 3 reviews ✓
[15/80] Fetching reviews for S1xnXRVFwH...   Found 3 reviews ✓
[16/80] Fetching

In [17]:
# Display the reviews dataframe
if len(reviews_df_accept_reject) > 0:
    print("=" * 60)
    print("REVIEWS DATAFRAME (Accept Reject)")
    print("=" * 60)
    print(f"\nTotal reviews: {len(reviews_df_accept_reject)}")
    print(f"\nReviews per paper:")
    print(reviews_df_accept_reject['paper_id'].value_counts().sort_index())
    print(f"\n\nFirst few reviews:")
    print(reviews_df_accept_reject.head(10).to_string())
else:
    print("No reviews to display")

REVIEWS DATAFRAME (Accept Reject)

Total reviews: 291

Reviews per paper:
paper_id
-59_mb1lOf4    4
0N8jUH4JMv6    3
0RDcd5Axok     3
3vzguDiEOr     3
5Jq1ASp33L     3
              ..
vVjIW3sEc1s    5
w2mYg3d0eot    4
x6x7FWFNZpg    4
xBeGd7sAND     4
ytZIYmztET     4
Name: count, Length: 80, dtype: int64


First few reviews:
     paper_id   review_id      review_type          rating summary strengths weaknesses questions comment                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [18]:
print("PERCENTAGE OF FEATURES WITH EMPTY STRINGS (ACCEPT REJECT)")
pd.DataFrame({
    "null_count": reviews_df_accept_reject.isna().sum(),
    "empty_string_count": (reviews_df_accept_reject == "").sum()
}).assign(
    pct_null=lambda x: x["null_count"] / len(reviews_df_accept_reject),
    pct_empty=lambda x: x["empty_string_count"] / len(reviews_df_accept_reject)
)

PERCENTAGE OF FEATURES WITH EMPTY STRINGS (ACCEPT REJECT)


,null_count,empty_string_count,pct_null,pct_empty
paper_id,0,0,0.0,0.000000
review_id,0,0,0.0,0.000000
review_type,0,0,0.0,0.000000
rating,0,149,0.0,0.512027
summary,0,291,0.0,1.000000
strengths,0,291,0.0,1.000000
weaknesses,0,291,0.0,1.000000
questions,0,291,0.0,1.000000
comment,0,291,0.0,1.000000
review,0,149,0.0,0.512027


In [19]:
# Display the reviews dataframe
if len(reviews_df_with_withdrawn) > 0:
    print("=" * 60)
    print("REVIEWS DATAFRAME (With Withdrawn)")
    print("=" * 60)
    print(f"\nTotal reviews: {len(reviews_df_with_withdrawn)}")
    print(f"\nReviews per paper:")
    print(reviews_df_with_withdrawn['paper_id'].value_counts().sort_index())
    print(f"\n\nFirst few reviews:")
    print(reviews_df_with_withdrawn.head(10).to_string())
else:
    print("No reviews to display")

REVIEWS DATAFRAME (With Withdrawn)

Total reviews: 285

Reviews per paper:
paper_id
-3yxxvDis3L    4
-59_mb1lOf4    4
0RDcd5Axok     3
1fLunL_hDj_    4
3Jf4Fr2I4T2    4
              ..
vlcVTDaufN     5
x6x7FWFNZpg    4
xBeGd7sAND     4
xxWl2oEvP2h    4
ytZIYmztET     4
Name: count, Length: 78, dtype: int64


First few reviews:
     paper_id   review_id      review_type          rating summary strengths weaknesses questions comment                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [20]:
print("PERCENTAGE OF FEATURES WITH EMPTY STRINGS (WITHDRAWN)")
pd.DataFrame({
    "null_count": reviews_df_with_withdrawn.isna().sum(),
    "empty_string_count": (reviews_df_with_withdrawn == "").sum()
}).assign(
    pct_null=lambda x: x["null_count"] / len(reviews_df_with_withdrawn),
    pct_empty=lambda x: x["empty_string_count"] / len(reviews_df_with_withdrawn)
)

PERCENTAGE OF FEATURES WITH EMPTY STRINGS (WITHDRAWN)


,null_count,empty_string_count,pct_null,pct_empty
paper_id,0,0,0.0,0.000000
review_id,0,0,0.0,0.000000
review_type,0,0,0.0,0.000000
rating,0,147,0.0,0.515789
summary,0,285,0.0,1.000000
strengths,0,285,0.0,1.000000
weaknesses,0,285,0.0,1.000000
questions,0,285,0.0,1.000000
comment,0,285,0.0,1.000000
review,0,147,0.0,0.515789


In [21]:
print(reviews_df_accept_reject['paper_id'].value_counts().describe())
print(reviews_df_with_withdrawn['paper_id'].value_counts().describe())

count    80.000000
mean      3.637500
std       0.641275
min       3.000000
25%       3.000000
50%       4.000000
75%       4.000000
max       5.000000
Name: count, dtype: float64
count    78.000000
mean      3.653846
std       0.620576
min       3.000000
25%       3.000000
50%       4.000000
75%       4.000000
max       5.000000
Name: count, dtype: float64


In [22]:
print("=" * 60)
print("WITHDRAWN PAPERS CHECK")
print("=" * 60)

withdrawn_samples = sampled_df_with_withdrawn.loc[sampled_df_with_withdrawn['decision'] == "Withdrawn"]
withdrawn_ids = withdrawn_samples["id"].tolist()
withdrawn_reviews = reviews_df_with_withdrawn.loc[reviews_df_with_withdrawn['paper_id'].isin(withdrawn_ids)]
print(f"{len(withdrawn_reviews)} reviews found for {len(withdrawn_ids)} withdrawn papers found in sample with withdrawn")
print(withdrawn_samples.head())
print(withdrawn_reviews.head())

WITHDRAWN PAPERS CHECK
51 reviews found for 16 withdrawn papers found in sample with withdrawn
    year           id                                              title  \
8   2020   H1lqSC4YvB          Generalized Transformation-based Gradient   
9   2020   HyxjWANFwH              Deep Learning-Based Average Consensus   
18  2020   B1eYGkBKDB  Fully Quantized Transformer for Improved Trans...   
19  2020   BJepraEFPr     Attention over Parameters for Dialogue Systems   
28  2021  3Jf4Fr2I4T2  Uncertainty Quantification for Bayesian Optimi...   

                                             abstract  \
8   The reparameterization trick has become one of...   
9   In this paper, we study the problem of acceler...   
18  State-of-the-art neural machine translation me...   
19  Dialogue systems require a great deal of diffe...   
28  Bayesian optimization is a class of global opt...   

                                              authors  \
8                                           Anba

In [23]:
print("PERCENTAGE OF FEATURES WITH EMPTY STRINGS (AMONG WITHDRAWN PAPERS ONLY)")

pd.DataFrame({
    "null_count": withdrawn_reviews.isna().sum(),
    "empty_string_count": (withdrawn_reviews == "").sum()
}).assign(
    pct_null=lambda x: x["null_count"] / len(withdrawn_reviews),
    pct_empty=lambda x: x["empty_string_count"] / len(withdrawn_reviews)
)

PERCENTAGE OF FEATURES WITH EMPTY STRINGS (AMONG WITHDRAWN PAPERS ONLY)


,null_count,empty_string_count,pct_null,pct_empty
paper_id,0,0,0.0,0.000000
review_id,0,0,0.0,0.000000
review_type,0,0,0.0,0.000000
rating,0,27,0.0,0.529412
summary,0,51,0.0,1.000000
strengths,0,51,0.0,1.000000
weaknesses,0,51,0.0,1.000000
questions,0,51,0.0,1.000000
comment,0,51,0.0,1.000000
review,0,27,0.0,0.529412
